# Download Tennessee Eastman Process Data

This notebook downloads the four original `.RData` files from the public Harvard Dataverse release into `data/raw/TEP/`. The files total about 1.4 GB (1.31 GiB).

Dataset: Rieth, C. A., Amsel, B. D., Tran, R., & Cook, M. B. (2017). *Additional Tennessee Eastman Process Simulation Data for Anomaly Detection Evaluation*, Version 1.0. Harvard Dataverse. https://doi.org/10.7910/DVN/6C3JR1

In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from hashlib import md5
from pathlib import Path
from urllib.request import Request, urlopen


def find_project_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "environment.yml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "TEP"
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
DATAVERSE_API = "https://dataverse.harvard.edu/api/access/datafile"

DATA_FILES = [
    {
        "filename": "TEP_FaultFree_Testing.RData",
        "file_id": 3031240,
        "size": 47_327_663,
        "md5": "38ad9810fc871026157086ae2c2f0ee9",
    },
    {
        "filename": "TEP_FaultFree_Training.RData",
        "file_id": 3031241,
        "size": 24_678_017,
        "md5": "ec126484534331f85001d8c4ebce6d17",
    },
    {
        "filename": "TEP_Faulty_Testing.RData",
        "file_id": 3031243,
        "size": 836_882_037,
        "md5": "556bdb64c83021bc0c5f92e427753565",
    },
    {
        "filename": "TEP_Faulty_Training.RData",
        "file_id": 3031242,
        "size": 494_063_194,
        "md5": "c5f594d54c47e620ff877feb58407fda",
    },
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_PATH}")
print(f"Download size: {sum(item['size'] for item in DATA_FILES):,} bytes")

Project root: <repo>
Raw data directory: <repo>\data\raw\TEP
Download size: 1,402,950,911 bytes


In [3]:
def file_md5(path: Path) -> str:
    checksum = md5()
    with path.open("rb") as data_file:
        for chunk in iter(lambda: data_file.read(8 * 1024 * 1024), b""):
            checksum.update(chunk)
    return checksum.hexdigest()


def is_verified(path: Path, file_info: dict) -> bool:
    return (
        path.is_file()
        and path.stat().st_size == file_info["size"]
        and file_md5(path) == file_info["md5"]
    )


def download_file(file_info: dict) -> None:
    filename = file_info["filename"]
    destination = RAW_DATA_PATH / filename
    partial_file = RAW_DATA_PATH / f"{filename}.part"

    if is_verified(destination, file_info):
        print(f"Already verified, skipping: {filename}")
        return

    if partial_file.exists() and partial_file.stat().st_size > file_info["size"]:
        partial_file.unlink()

    offset = partial_file.stat().st_size if partial_file.exists() else 0
    if offset == file_info["size"]:
        if file_md5(partial_file) == file_info["md5"]:
            partial_file.replace(destination)
            print(f"Verified completed download: {filename}")
            return
        partial_file.unlink()
        offset = 0

    request_headers = {"User-Agent": "carl-data-download-notebook/1.0"}
    if offset:
        request_headers["Range"] = f"bytes={offset}-"
    request = Request(
        f"{DATAVERSE_API}/{file_info['file_id']}",
        headers=request_headers,
    )
    response = urlopen(request, timeout=60)

    if offset and response.status == 206:
        content_range = response.headers.get("Content-Range", "")
        if not content_range.startswith(f"bytes {offset}-"):
            response.close()
            raise RuntimeError(f"Unexpected Content-Range while resuming {filename}: {content_range}")
        write_mode = "ab"
    else:
        offset = 0
        write_mode = "wb"

    bytes_written = offset
    next_report_percent = ((offset * 100 // file_info["size"] // 10) + 1) * 10
    print(f"Downloading {filename} (resume at {offset:,} bytes)")
    with response, partial_file.open(write_mode) as output_file:
        while chunk := response.read(8 * 1024 * 1024):
            output_file.write(chunk)
            bytes_written += len(chunk)
            percent = bytes_written * 100 // file_info["size"]
            if percent >= next_report_percent:
                print(f"  {filename}: {percent}%", flush=True)
                next_report_percent = (percent // 10 + 1) * 10

    actual_size = partial_file.stat().st_size
    if actual_size != file_info["size"]:
        raise RuntimeError(
            f"Incomplete download for {filename}: expected {file_info['size']:,} bytes, got {actual_size:,}. "
            "The partial file was kept; rerun this cell to resume."
        )

    actual_md5 = file_md5(partial_file)
    if actual_md5 != file_info["md5"]:
        partial_file.unlink()
        raise ValueError(
            f"Checksum mismatch for {filename}: expected {file_info['md5']}, got {actual_md5}. "
            "The incomplete or corrupt download was removed; rerun this cell to try again."
        )

    partial_file.replace(destination)
    print(f"Verified {filename} ({actual_size:,} bytes, MD5 {actual_md5})")

In [4]:
for file_info in DATA_FILES:
    download_file(file_info)

print("All TEP raw data files are present and verified.")

  TEP_FaultFree_Testing.RData: 17%
  TEP_FaultFree_Testing.RData: 35%
  TEP_FaultFree_Testing.RData: 53%
  TEP_FaultFree_Testing.RData: 70%
  TEP_FaultFree_Testing.RData: 88%
  TEP_FaultFree_Testing.RData: 100%
Verified TEP_FaultFree_Testing.RData (47,327,663 bytes, MD5 38ad9810fc871026157086ae2c2f0ee9)
  TEP_FaultFree_Training.RData: 33%
  TEP_FaultFree_Training.RData: 67%
  TEP_FaultFree_Training.RData: 100%
Verified TEP_FaultFree_Training.RData (24,678,017 bytes, MD5 ec126484534331f85001d8c4ebce6d17)
  TEP_Faulty_Testing.RData: 10%
  TEP_Faulty_Testing.RData: 20%
  TEP_Faulty_Testing.RData: 30%
  TEP_Faulty_Testing.RData: 40%
  TEP_Faulty_Testing.RData: 50%
  TEP_Faulty_Testing.RData: 60%
  TEP_Faulty_Testing.RData: 70%
  TEP_Faulty_Testing.RData: 80%
  TEP_Faulty_Testing.RData: 90%
  TEP_Faulty_Testing.RData: 100%
Verified TEP_Faulty_Testing.RData (836,882,037 bytes, MD5 556bdb64c83021bc0c5f92e427753565)
  TEP_Faulty_Training.RData: 10%
  TEP_Faulty_Training.RData: 20%
  TEP_Faulty